In [94]:
import pandas as pd
from time import time
from kafka import KafkaProducer
from models_green import green_trip_from_row

In [95]:



# Only read the columns we need
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime', 
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

df = pd.read_parquet('green_tripdata_2025-10.parquet', columns=columns)

print(f"Loaded {len(df)} rows")
print(df.head())

Loaded 49416 rows
  lpep_pickup_datetime lpep_dropoff_datetime  PULocationID  DOLocationID  \
0  2025-10-01 00:21:47   2025-10-01 00:24:37           247            69   
1  2025-10-01 00:14:03   2025-10-01 00:24:14            66            25   
2  2025-10-01 00:16:44   2025-10-01 00:16:47           244           244   
3  2025-10-01 00:07:36   2025-10-01 00:32:14            95           170   
4  2025-09-30 21:10:29   2025-09-30 21:22:30            82           138   

   passenger_count  trip_distance  tip_amount  total_amount  
0              1.0           0.70        1.70         10.00  
1              1.0           1.61        2.78         16.68  
2              1.0           0.00        2.20         13.20  
3              1.0          10.37       11.31         67.85  
4              1.0           4.07        6.82         34.12  


In [96]:
server = 'localhost:9092'
topic_name = 'green-trips'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=green_trip_serializer
)

print(f"Producer connected to {server}")

Producer connected to localhost:9092


In [97]:
from time import time

t0 = time()

for idx, row in df.iterrows():
    trip = green_trip_from_row(row)
    producer.send(topic_name, value=trip)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 2.99 seconds
